# AI Agent vs. Human Detection: CUDA-Accelerated Evaluation

## 1. Introduction

This notebook evaluates pre-trained machine learning models using pre-split training, testing, and evaluation datasets. It is optimized for CUDA to utilize GPU acceleration.

## 2. Setup and CUDA Check

In [18]:
%pip install scikit-learn matplotlib seaborn torch pytorch_tabnet

Note: you may need to restart the kernel to use updated packages.


In [17]:
import pandas as pd
import numpy as np
import pickle
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from pytorch_tabnet import TabnetRegressor
from rtdl_revisiting_models import FTTransformer
from sklearn.metrics import f1_score, accuracy_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor
# from sklearn.datasets import make_regression
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, mean_squared_error, r2_score

# Check for CUDA availability
device = torch.device("cuda") if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if device == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")

ModuleNotFoundError: No module named 'pytorch_tabnet'

## 3. Loading Pre-split Data

In [13]:
def load_pickle(file_path):
    with open(file_path, 'rb') as f:
        return pickle.load(f)

# Replace with your actual pre-split pickle file paths
train_data = load_pickle('processed_v1_features_train.pkl')
test_data = load_pickle('processed_v1_features_test.pkl')
eval_data = load_pickle('processed_v1_features_val.pkl')

# Assuming the pickles are dictionaries or tuples containing (X, y)
print(f"train_data type: {type(train_data)}")
print(f"train_data shape: {train_data.shape if hasattr(train_data, 'shape') else 'N/A'}")
print(f"train_data ndim: {train_data.ndim if hasattr(train_data, 'ndim') else 'N/A'}")
print(f"train_data dtype: {train_data.dtype if hasattr(train_data, 'dtype') else 'N/A'}")

# X_train, y_train = train_data['X'], train_data['y']
# X_test, y_test = test_data['X'], test_data['y']
# X_eval, y_eval = eval_data['X'], eval_data['y']
# X_train, y_train = train_data
# X_test, y_test = test_data
# X_eval, y_eval = eval_data

# print(f"Data Loaded: Train({len(X_train)}), Test({len(X_test)}), Eval({len(X_eval)})")


train_data type: <class 'numpy.ndarray'>
train_data shape: (24514, 773)
train_data ndim: 2
train_data dtype: float64


In [14]:
# 1. Initialize the Regressor instead of the Classifier
# n_estimators: number of trees
# n_jobs=-1: uses all your CPU cores for faster training
rf_regressor = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)

# 2. Train the model
print("Training the Random Forest Regressor...")
rf_regressor.fit(X_train, y_train)

# 3. Evaluate on the Validation (Eval) set
y_pred_eval = rf_regressor.predict(X_eval)

# For regression, we use MSE and R-squared
mse = mean_squared_error(y_eval, y_pred_eval)
r2 = r2_score(y_eval, y_pred_eval)

print(f"Mean Squared Error: {mse:.4f}")
print(f"R-squared Score: {r2:.4f}")

Training the Random Forest Regressor...
Mean Squared Error: 0.0097
R-squared Score: 0.6267


In [16]:
# 1. Initialise the Multi Layer Perceptron algorithm
mlp = MLPRegressor(hidden_layer_sizes=(100, 50), max_iter=1000, random_state=42)

# 2. Train the model
print("Training the Multi Layer Perceptron...")
mlp.fit(X_train, y_train)

# 3. Evaluate Validation set
y_pred_eval = mlp.predict(X_eval)
r2 = r2_score(y_eval, y_pred_eval)

print(f"R-squared Score: {r2:.4f}")

Training the Multi Layer Perceptron...
R-squared Score: 0.8932


In [ ]:
import numpy as np
from sklearn.ensemble import RandomForestClassifier # Or RandomForestRegressor
from sklearn.metrics import accuracy_score, classification_report

# 1. Slicing the X and y vectors
# [:, :-1] selects all rows and all columns EXCEPT the last one
# [:, -1] selects all rows and ONLY the last column
X_train, y_train = train_data[:, :-1], train_data[:, -1]
X_test, y_test = test_data[:, :-1], test_data[:, -1]
X_eval, y_eval = eval_data[:, :-1], eval_data[:, -1]

print(f"Feature shape: {X_train.shape}") # Should be (24514, 772)
print(f"Target shape: {y_train.shape}")  # Should be (24514,)

# 2. Initialize the Random Forest
# n_estimators is the number of trees; random_state ensures reproducibility
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)

# 3. Train the model
print("Training the Random Forest...")
rf_model.fit(X_train, y_train)

# 4. Evaluate on the Validation (Eval) set
y_pred_eval = rf_model.predict(X_eval)
print(f"Validation Accuracy: {accuracy_score(y_eval, y_pred_eval):.4f}")

Feature shape: (24514, 772)
Target shape: (24514,)
Training the Random Forest...


ValueError: Unknown label type: continuous. Maybe you are trying to fit a classifier, which expects discrete classes on a regression target with continuous values.

In [5]:
# Check the structure of loaded data
print(f"train_data type: {type(train_data)}")
print(f"train_data keys/structure: {train_data.keys() if isinstance(train_data, dict) else train_data if isinstance(train_data, tuple) else 'unknown'}")

# If it's a dict, show available keys
if isinstance(train_data, dict):
    print(f"Available keys: {list(train_data.keys())}")
    # Check the first key's contents
    first_key = list(train_data.keys())[0]
    print(f"Sample of first key: {type(train_data[first_key])}")

train_data type: <class 'numpy.ndarray'>
train_data keys/structure: unknown


## 4. Loading and Running Models on CUDA

In [ ]:
# Note: For Scikit-learn models (Random Forest, etc.), CUDA is typically used via libraries like cuML or by using specific GPU-enabled implementations (e.g., XGBoost, CatBoost).
# If the models are PyTorch-based, we move them to the device.

def run_model_on_cuda(model_path, X_data):
    model = load_pickle(model_path)
    
    # If the model is a PyTorch model
    if hasattr(model, 'to'):
        model = model.to(device)
        X_tensor = torch.tensor(X_data.values if hasattr(X_data, 'values') else X_data).float().to(device)
        with torch.no_grad():
            preds = model(X_tensor)
            return preds.cpu().numpy()
    
    # If the model is an XGBoost/CatBoost model with GPU support
    # (These usually handle device placement internally if configured)
    return model.predict(X_data)

# Paths to your model pickles
model_paths = {
    'Model_1': 'model_1.pkl',
    'Model_2': 'model_2.pkl'
}

results = []
for name, path in model_paths.items():
    print(f"Evaluating {name}...")
    y_pred = run_model_on_cuda(path, X_eval)
    
    # Handle probabilistic outputs if necessary
    if len(y_pred.shape) > 1 and y_pred.shape[1] > 1:
        y_pred = np.argmax(y_pred, axis=1)
    
    f1 = f1_score(y_eval, y_pred)
    results.append({'Model': name, 'F1 Score': f1})

eval_df = pd.DataFrame(results)
print(eval_df)
```

## 5. Final Comparison and Visualization

In [ ]:
plt.figure(figsize=(10, 6))
sns.barplot(data=eval_df, x='Model', y='F1 Score')
plt.title('Model F1 Score Comparison (CUDA Accelerated)')
plt.show()
```